In [ ]:
import os
from dotenv import load_dotenv
from google.genai.client import Client 
from google import genai
from google.genai.types import GenerateContentConfig, ThinkingConfig
from pprint import pprint
from pydantic import BaseModel, Field
from enum import Enum
from typing import Callable, Dict, Any
import re , json
from datetime import datetime


Let's first import our Gemini api key 

In [2]:
load_dotenv()
val = os.getenv("GEMINI_API_KEY")
model = "gemini-2.5-flash"
googleClient = genai.Client()


In [ ]:
resp = googleClient.models.generate_content(
    model= model,
    contents="Explique moi le théorème de Rolle"
)

resp


Now let's define the types that we will need to define our agent . As explained an agent should receive and output messages , but a message could have multiple types. It can be a User message which is the prompt of the user , it can be a System message which is the role that we give to our agent and how he should behave and finally the Agent message which is his answer .

In [83]:
class MessageType(Enum):
    USER = "user"
    AGENT = "agent"
    
class Message(BaseModel):
    type: MessageType
    content: str
    def to_gemini_format(self) -> dict:
        """Convert the message to the message that can be read by Gemini"""      
        if self.type == MessageType.USER:
            role = "user"
        elif self.type == MessageType.AGENT:
            role = "model"
        else:
            raise ValueError(f"Type de message inconnu : {self.type}")

        return {
            "role": role,
            "parts": [{"text": self.content}]
        }
    
    

# I) ReAct Agent

Here we want to let the user initialize an agent with a Gemini Client and gives him a system prompt where we precise that the agent needs to output the correct tool to call at each step to execute it. 
The strategy is to let the user add tools to his agent , see the Chain of thoughts of the agent and at every step execute the function needed by the agent and returning it the result and so on .

In [ ]:
class Agent:
    def __init__(self, client: Client, system_prompt : str| None = None):
        self.client = client
        self.system_prompt = system_prompt or "Tu es un agent ReAct intelligent et précis."
        self.messages: list[Message] = []
        self.config = None
        
        #Tool definition
        self.tools : Dict[str,Callable] = {}
        self.tool_descriptions: list[Dict] = []

    def __call__(self, message: str | None = None) -> str :
        """ When an user call an agent with a question it executes this function 

        Args:
            message (str | None, optional): User message. Defaults to None.

        Returns:
            str: Agent answer
        """
        if message:
            self.messages.append(Message(type=MessageType.USER, content=message))

        result = self.execute()        
        self.messages.append(Message(type
                                     =MessageType.AGENT, content=result))
        return result

    def execute(self):
        history = [m.to_gemini_format() for m in self.messages]

        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=history,
            config=self.config          
        )
        return response.text

    def add_tool(self, function: Callable, description: str = None):
        #Take the function name
        tool_name = function.__name__
        if tool_name in self.tools:
            print(f"WARNING: Tool '{tool_name} already present !")
            return 
        # Add the tool to the existing tools
        self.tools[tool_name] = function 
        # Add the description of the tool
        desc = description or function.__doc__ or f"Call the function {tool_name}"
        self.tool_descriptions.append({"name":tool_name, "description":desc.strip()})
        self._update_config()
        
    def _update_config(self):
        """Update the config with the new tools"""
        
        system_text = self._build_system_prompt()
        self.config = GenerateContentConfig(
            system_instruction=system_text
        )

    
    def _build_system_prompt(self) -> str:
        text_tools = "\n\nAvailable tools:\n"
        for tool in self.tool_descriptions:
            text_tools += f"- {tool['name']}: {tool['description']}\n"
            
        base_prompt = f"""
        
        Tu es un agent sympa qui répond à la question de l utilisateur.\n
        Tu as le droit d utiliser UNIQUEMENT LES TOOLS DISPONNIBLES:\ng
        {text_tools}\n
        
        A chaque appel tu répondras avec exactement cette structure:
        Tought : Ta reflexion
        Action : nom du tool que tu veux utiliser
        Action inputs: UNIQUEMENT les argument pour le tool\n
        
        - Si tu as suffisamment d'informations pour répondre à l'utilisateur, utilise :
        Thought: ...
        Final Answer: ta réponse complète et naturelle à l'utilisateur
        Ne parle jamais du format dans ta réponse finale. Sois direct , concis et clair.
        """
        
        return base_prompt
        
    
    def get_tools(self):
        return self.tool_descriptions
    
    def get_history(self):
        return self.messages
    
    def execute_tool(self,tool_name:str, tool_input:Any) -> str:
            if tool_name not in self.tools:
                return f"Erreur: L'outil '{tool_name}' n'existe pas."
            try:
                func = self.tools[tool_name]
                if isinstance(tool_input,dict):
                    result = func(**tool_input)
                else:
                    result = func(tool_input)
                return str(result)
            except Exception as e:
                return f"Erreur lors de l'exécution de {tool_name}: {str(e)}"
            
    def _parse_response(self, text: str) -> dict:
        text = text.strip()

        # Extraction du Thought (toujours présent)
        thought_match = re.search(r'Thought[:\s]*(.*?)(?=Action:|Final Answer:|$)', 
                                  text, re.DOTALL | re.IGNORECASE)
        thought = thought_match.group(1).strip() if thought_match else ""

        # Détection Final Answer
        final_match = re.search(r'Final Answer[:\s]*(.*)', text, re.DOTALL | re.IGNORECASE)
        if final_match:
            return {
                "type": "final", 
                "thought": thought,
                "content": final_match.group(1).strip()
            }

        # Détection Action
        action_match = re.search(r'Action[:\s]*(\w+)', text, re.IGNORECASE)
        if action_match:
            tool_name = action_match.group(1).strip()

            input_match = re.search(r'Action Input[:\s]*(.*)', text, re.DOTALL | re.IGNORECASE)
            input_str = input_match.group(1).strip() if input_match else ""

            # Nettoyage du input (enlève les préfixes parasites)
            input_str = re.sub(r'^(s:|json:|input:)\s*', '', input_str, flags=re.IGNORECASE).strip()

            # Essayer de parser JSON
            try:
                if input_str.startswith('{') and input_str.endswith('}'):
                    tool_input = json.loads(input_str)
                else:
                    tool_input = input_str
            except:
                tool_input = input_str

            return {
                "type": "action",
                "thought": thought,
                "tool_name": tool_name,
                "tool_input": tool_input
            }

        # Cas par défaut
        return {
            "type": "text",
            "thought": thought,
            "content": text
        }
        
        
    def run(self, user_query: str, max_steps: int = 12) -> str:
        print("=== Début du ReAct ===\n")
        self.messages.append(Message(type=MessageType.USER, content=user_query))

        for step in range(max_steps):
            self._update_config()

            response_text = self.execute()
            self.messages.append(Message(type=MessageType.AGENT, content=response_text))

            parsed = self._parse_response(response_text)

            # Affichage du Thought
            print(f"--- Étape {step + 1} ---")
            if parsed.get("thought"):
                print(f"Thought: {parsed['thought']}")

            if parsed["type"] == "final":
                print(f"✅ Réponse finale obtenue en {step + 1} étapes\n")
                print(f"Final Answer: {parsed['content']}")
                return parsed["content"]

            elif parsed["type"] == "action":
                tool_name = parsed["tool_name"]
                tool_input = parsed["tool_input"]
                print(f"Action: {tool_name}")
                print(f"Action Input: {tool_input}\n")

                observation = self.execute_tool(tool_name, tool_input)
                print(f"Observation: {observation}\n")

                self.messages.append(
                    Message(type=MessageType.USER, content=f"Observation: {observation}")
                )

            else:
                print(f"Réponse texte : {parsed.get('content', response_text)}")
                return response_text

        print("Limite de steps atteinte.")
        return "Limite atteinte sans réponse finale."

In [ ]:
def get_weather(city: str) -> str:
    """Retourne la météo actuelle d'une ville (simulation)"""
    print(f"[TOOL] get_weather appelé avec city = {city}")
    weather_data = {
        "Paris": "18°C, ensoleillé",
        "Lyon": "15°C, pluie légère",
        "Marseille": "22°C, ciel dégagé",
        "Toulouse": "17°C, nuageux"
    }
    return weather_data.get(city, f"18°C à {city} (simulation)")


def get_population(city: str) -> str:
    """Retourne la population d'une ville (simulation)"""
    print(f"[TOOL] get_population appelé avec city = {city}")
    pop_data = {
        "Paris": "Environ 2 140 000 habitants",
        "Lyon": "Environ 520 000 habitants",
        "Marseille": "Environ 870 000 habitants",
        "Toulouse": "Environ 500 000 habitants"
    }
    return pop_data.get(city, f"Population de {city} inconnue")


def get_current_time() -> str:
    """Retourne l'heure actuelle en France (Europe/Paris)"""
    now = datetime.now()
    return now.strftime("Il est %H:%M:%S - %A %d %B %Y")

In [ ]:
# Création de l'agent
agent = Agent(
    client=googleClient,
    system_prompt="Tu es un assistant ReAct très intelligent qui aide l'utilisateur."
)

# Ajout des tools
agent.add_tool(get_weather)
agent.add_tool(get_population)

# Lancement du ReAct
print("=== Lancement de l'agent ===\n")
final_answer = agent.run("Quelle est la météo à Paris et quelle est sa population et quelle heure est il actuellement ?")

print("\n" + "="*50)
print("RÉPONSE FINALE :")
print(final_answer)

=== Lancement de l'agent ===

=== Début du ReAct ===

--- Étape 1 ---
Thought: L'utilisateur demande la météo et la population de Paris. Je dois d'abord appeler le tool `get_weather` pour obtenir la météo.
Action: get_weather
Action Input: {'city': 'Paris'}

[TOOL] get_weather appelé avec city = Paris
Observation: 18°C, ensoleillé

--- Étape 2 ---
Thought: J'ai obtenu la météo de Paris. Maintenant, je dois obtenir la population de Paris en utilisant le tool `get_population`.
Action: get_population
Action Input: {'city': 'Paris'}

[TOOL] get_population appelé avec city = Paris
Observation: Environ 2 140 000 habitants

--- Étape 3 ---
Thought: J'ai obtenu toutes les informations nécessaires : la météo et la population de Paris. Je peux maintenant formuler la réponse finale.
✅ Réponse finale obtenue en 3 étapes

Final Answer: À Paris, il fait actuellement 18°C et le temps est ensoleillé. La ville compte environ 2 140 000 habitants.

RÉPONSE FINALE :
À Paris, il fait actuellement 18°C et l